# Longitudinal Analysis of the Allegheny County Jail Oversight Board Meeting Minutes

- Contributor: Anna Ringwood
- AI Acknowledgements: The following code was written with assistance from GitHub Copilot.

## Text Preprocessing — Processing Text from JOB Minutes

This notebook contains code for ingesting the scraped text files and beginning to further extract only relevant text information.

In [2]:
import csv
import time
from pathlib import Path
from urllib.parse import urlparse

In [3]:
# Set up paths, similar to how we did in the preceding file
BASE = Path("..").resolve()
OUT_DIR = BASE / "Data" / "Text" # where the raw text files are kept

In [69]:
raw_docs = []
doc_names = []

# This section's code taken from last block of preceding file with minor modifications to variable names
for path in sorted(OUT_DIR.glob("*.txt")):
    raw_docs.append(path.read_text(encoding="utf-8"))
    doc_names.append(path.name)

print(f"{len(raw_docs)} document(s) → lists `documents` (text) and `doc_names` (filenames)")

# Remove all instances of "\n" since spaCy doesn't handle them very well (and we don't need them because we're tokenizing on words).
# Also replace any instances of more than one space with a single space
prelim_docs = []
for doc in raw_docs:
    temp_doc = doc.replace("\n", " ") # replace newlines with spaces
    temp_doc = " ".join(temp_doc.split()) # replace multiple spaces with a single space
    prelim_docs.append(temp_doc)

156 document(s) → lists `documents` (text) and `doc_names` (filenames)


In [70]:
raw_docs[115]

'1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n11\n12\n13\n14\n15\n16\n17\n18\n19\n20\n21\n22\n23\n24\n25\nGALVIN REPORTING SERVICES\n412-897-2010  -- 412-461-1838 (FAX)\n1\nALLEGHENY  COUNTY\nJAIL OVERSIGHT  BOARD  MEETING\nThursday\nOctober  5, 2023\nGold Room\n4th Floor\nAllegheny  County  Courthouse\n436 Grant  Street\nPittsburgh , Pennsylvania  15219\n\n1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n11\n12\n13\n14\n15\n16\n17\n18\n19\n20\n21\n22\n23\n24\n25\nGALVIN REPORTING SERVICES\n412-897-2010  -- 412-461-1838 (FAX)\n2\nMEMBERS OF THE BOARD IN ATTENDANCE: \nJudge Elliot Howsie\nBethany Hallam for County Council President\n Pat Catena\nStephen Pilarski for County Executive\nRichard Fitzgerald\nController Corey O\'Connor\nSheriff Kevin Kraus\nTerri Klein\nM. Gayle Moss\nJAIL ADMINISTRATION IN ATTENDANCE: \nInterim Warden Shane T. Dady\nChief Deputy Warden Jason Beasom\nHSA Dr. Ashley Brinkman\nChief Deputy Warden Blythe Toma Chief \nDeputy Warden Connie Clark\n\n1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n11\n12\n13\n14\n

In [71]:
prelim_docs[115]

'1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 GALVIN REPORTING SERVICES 412-897-2010 -- 412-461-1838 (FAX) 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING Thursday October 5, 2023 Gold Room 4th Floor Allegheny County Courthouse 436 Grant Street Pittsburgh , Pennsylvania 15219 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 GALVIN REPORTING SERVICES 412-897-2010 -- 412-461-1838 (FAX) 2 MEMBERS OF THE BOARD IN ATTENDANCE: Judge Elliot Howsie Bethany Hallam for County Council President Pat Catena Stephen Pilarski for County Executive Richard Fitzgerald Controller Corey O\'Connor Sheriff Kevin Kraus Terri Klein M. Gayle Moss JAIL ADMINISTRATION IN ATTENDANCE: Interim Warden Shane T. Dady Chief Deputy Warden Jason Beasom HSA Dr. Ashley Brinkman Chief Deputy Warden Blythe Toma Chief Deputy Warden Connie Clark 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 GALVIN REPORTING SERVICES 412-897-2010 -- 412-461-1838 (FAX) 3 COMMUNITY CORRECT

The newline characters have been replaced with spaces and there are no multi-spaces, so we can use spaCy now to conduct additional preprocessing

In [72]:
import spacy
nlp = spacy.load('en_core_web_sm', disable = ['parser', 'ner']) # disable parser and NER so package runs faster

In [74]:
from tqdm import tqdm

documents = []
for prelim_doc in tqdm(prelim_docs): # use tqdm for progress monitoring; time varies depending on your machine: ~4-15 min
    documents.append(nlp(prelim_doc))

100%|██████████| 156/156 [02:53<00:00,  1.11s/it]


In [76]:
# Create a helper function to convert document names to their indices when needed
def doc_name_to_idx(name):
    assert name in doc_names, "Document name not found"
    for i, doc_name in enumerate(doc_names):
        if doc_name == name:
            print(f"Document {i} has the title {name}.")

# Test function
doc_name_to_idx("2022_5_0_JOB_Minutes.txt") # returns index 110
doc_names[110]

Document 110 has the title 2022_5_0_JOB_Minutes.txt.


'2022_5_0_JOB_Minutes.txt'

### Removing Unneeded Transcription Data

Now that the text files have been imported and initially processedby `spaCy`, let's clean them up further. For example, in more recent years the minutes are just transcripts with line numbers and transcription company information on each page. We don't need this data.

In [77]:
# Get indices of the documents that contain "GALVIN REPORTING SERVICES" a.k.a. the ones that contain transcription line numbers + company name we want to remove
transcription_junk = "1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 GALVIN REPORTING SERVICES 412-897-2010 -- 412-461-1838 (FAX)"
docs_w_transcription_junk = []

for i, doc in enumerate(documents):
    if "GALVIN REPORTING SERVICES" in doc.text:
        docs_w_transcription_junk.append(i)

In [78]:
# Remove all occurrences of the transcription junk from the documents
for i in docs_w_transcription_junk:
    documents[i] = documents[i].text.replace(transcription_junk, "")

In [79]:
[documents[i] for i in docs_w_transcription_junk[:10]] # check that junk was removed - scan of the first 10 documents show that the removal was successful

[' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING Thursday October 6, 2022 Gold Room 4th Floor Allegheny County Courthouse 436 Grant Street Pittsburgh , Pennsylvania 15219  2 MEMBERS OF THE BOARD: Judge Elliot Howsie Judge Beth Lazzara Bethany Hallam for Councilman Catena Stephen Pilarski for County Executive Richard Fitzgerald Controller Corey O\'Connor Chief Deputy John Kearney for Kevin Krauss Terri Klein M. Gayle Moss JAIL ADMINISTRATION IN ATTENDANCE: Warden Orlando Harper Chief Deputy Warden Jason Beasom HSA Dr. Ashley Brinkman Deputy Warden of Administrative Operations and Employee Development Blythe Toma Kevin Kordzi - Passages to Recover Adam Zak - The Renewal Center Steve Esswein - Electronic Monitoring  3 PUBLIC SPEAKERS : Nichole Remmert Reverend Richard Freeman Sister Barbara Finch Sharon Bonavoglia Janelle Tipton Elizabeth Schongar Brandi Fisher Marion Damick Tim Stevens Eric Miskovitch Alan Guenther Tanisha Long Kimberly Andrews Brian Englert Bobbi Linskens Kim Williams

### End Matter of Minutes

Additionally, the transcribed documents include end matter with vocab glossaries and indices, which would both throw off our analysis and take up unnecessary processing power during lemmatization. We therefore looked for keywords that would indicate the end of the transcription (e.g., "adjournment"). After some trial-and-error, we discovered that all of the transcripted documents except one contained a notary certification at the end, after which the glossary and index appeared. Thus, for these docs we removed all the text after the certification header. For the final document that didn't follow this pattern, we manually trimmed the end matter.

In [80]:
one_occurrence_idxs = []
gt_one_occurrence_idxs = []
lt_one_occurrence_idxs = []
word_phrase = "C E R T I F I C A T E"

for i in docs_w_transcription_junk:
    
    if documents[i].upper().count(word_phrase) > 1:
        gt_one_occurrence_idxs.append(i)
    elif documents[i].upper().count(word_phrase) < 1:
        lt_one_occurrence_idxs.append(i)
    else:
        one_occurrence_idxs.append(i)

print(f"One instance ({len(one_occurrence_idxs)}): {one_occurrence_idxs}")
print(f"Less than one instance ({len(lt_one_occurrence_idxs)}): {lt_one_occurrence_idxs}")
print(f"More than one instance ({len(gt_one_occurrence_idxs)}): {gt_one_occurrence_idxs}")

One instance (43): [103, 104, 105, 115, 116, 117, 118, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155]
Less than one instance (1): [119]
More than one instance (0): []


In [81]:
doc_names[119] # upon manual inspection, we'll remove everything after "I, Diane  G. Galvin , a court  reporter"

'2023_2_0_JOB_Minutes.txt'

In [82]:
# Remove all text after "C E R T I F I C A T E" in the documents
for i in one_occurrence_idxs:
    cert_idx = documents[i].upper().find(word_phrase)
    documents[i] = documents[i][:cert_idx] # keep everything up to but not including the word "C E R T I F I C A T E "

# For document 119, remove everything after "I, Diane  G. Galvin , a court  reporter"
reporter_idx = documents[119].upper().find("I, DIANE  G. GALVIN , A COURT  REPORTER")
documents[119] = documents[119][:reporter_idx] # keep everything up to but not including the reporter information

In [83]:
# Check that the end matter was successfully removed from all transctiption documents
for i in docs_w_transcription_junk:
    print(f"Document {i} ends with: {documents[i][-100:]}") # check the last 100 characters of the document to confirm that the end matter was removed


Document 103 ends with: OWSIE : So moved . The meeting is adjourned . The meeting adjourned at approximately 7:45 p.m.  197 
Document 104 ends with:  HOWSIE : We're adjourned . (Whereupon , the hearing was concluded at approximately 7:22 p.m.)  203 
Document 105 ends with: ry one. Stay safe. See you in the new year. (The meeting concluded at approximately 7:40 p.m.)  224 
Document 115 ends with: rn . JUDGE HOWSIE : Thank you. I second . (Whereupon , the hearing was concluded at 6:49 p.m.)  180 
Document 116 ends with: E : Thank you. The meeting is adjourned . (Whereupon , the meeting was adjourned at 6:49 p.m.)  163 
Document 117 ends with: all.  166 MS. HALLAM : Great job, Terri . (Whereupon , the hearing was adjourned at 6:47 p.m.)  167 
Document 118 ends with: KRAUS : Motion to adjourn . (Whereupon , the meeting was concluded at approximately 7:45 p.m.)  171 
Document 119 ends with: 3] - 17:20, 126:12, 128:3 Z Zankovich [1] - 44:6 zero [2] - 8:12, 83:16 Zilinek [1] - 43:9 Sincerely
